# 01 · Locked Text Teacher — Idiap InducT-GCN

Participant-disjoint DAIC-WOZ teacher export.

- TRAIN: 107 participants
- DEV: 34 participants
- participant 440 excluded
- TEST is never opened
- TRAIN outputs are in-sample checkpoint training-graph document nodes
- DEV outputs use the checkpoint saved `A_dev`


In [ ]:
from google.colab import drive,userdata
drive.mount('/content/drive')

from pathlib import Path
import base64,subprocess,sys,json
REPO='engrkhubabahmad/reliability-aware-depression-kd'
ROOT=Path('/content/reliability-aware-depression-kd')
token=userdata.get('GITHUB_TOKEN'); assert token,'Add GITHUB_TOKEN to Colab Secrets'
auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git=['git','-c',f'http.extraHeader=AUTHORIZATION: basic {auth}']
if not (ROOT/'.git').exists():
    subprocess.run(git+['clone',f'https://github.com/{REPO}.git',str(ROOT)],check=True)
subprocess.run(git+['-C',str(ROOT),'fetch','origin','main'],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','main'],check=True)
subprocess.run(git+['-C',str(ROOT),'pull','--ff-only','origin','main'],check=True)
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements-colab.txt')])
print('main:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())


## Author source

Keep the public Idiap repository in Drive so Colab restarts do not require downloading it again.


In [ ]:
SRC=Path('/content/drive/MyDrive/tools/solo_teacher_sources/bias_in_daic-woz')
SRC.parent.mkdir(parents=True,exist_ok=True)
if not (SRC/'.git').exists():
    subprocess.run(['git','clone','https://github.com/idiap/bias_in_daic-woz.git',str(SRC)],check=True)
print('Idiap source commit:',subprocess.check_output(['git','-C',str(SRC),'rev-parse','HEAD'],text=True).strip())
required=[
 'data/AVEC_16_data/train_IDS.txt','data/AVEC_16_data/dev_IDS.txt',
 'model/Participant/model_inductgcn[250].pkl','model/Participant/vtzer_inductgcn[250].pkl'
]
for p in required:
    assert (SRC/p).exists(),SRC/p
print('Required published assets: OK')


## Export TRAIN-107 and DEV-34 probabilities/logits


In [ ]:
OUT=Path('/content/drive/MyDrive/DAIC_WOZ/experiments/idiap_text_teacher')
subprocess.run([
 sys.executable,'-m','scripts.teachers.idiap_text.export_targets',
 '--daic-root','/content/drive/MyDrive/DAIC_WOZ',
 '--source-root',str(SRC),'--output',str(OUT)
],cwd=ROOT,check=True)


## Integrity check


In [ ]:
import pandas as pd,numpy as np
tr=pd.read_csv(OUT/'train_text_kd_targets.csv')
dv=pd.read_csv(OUT/'dev_text_predictions.csv')
assert len(tr)==107 and tr.participant_id.nunique()==107
assert len(dv)==34 and dv.participant_id.nunique()==34 and 440 not in set(dv.participant_id)
assert not set(tr.participant_id)&set(dv.participant_id)
for d in (tr,dv):
    q=1/(1+np.exp(-d.text_logit.to_numpy(float)))
    assert np.allclose(q,d.text_probability.to_numpy(float),atol=1e-6)
    assert d[['text_probability','text_logit']].notna().all().all()
audit=json.loads((OUT/'text_target_audit.json').read_text())
assert audit['test_opened'] is False
print(json.dumps(audit,indent=2))
print('\nTEXT TEACHER LOCKED: targets verified; TEST CLOSED.')
